# Scaling up: AI2D, ChartQA, TextVQA (free, no gating)

**Before running:**
- Kaggle: Settings (right panel) -> Accelerator -> **GPU T4 x2**, and **Internet -> On**.
- Colab: Runtime -> Change runtime type -> **GPU (T4)**.

This notebook extends the earlier MathVista/MMMU hybrid-score pipeline to three new,
completely free, ungated datasets, to build a real 5-dataset multi-shift stream:

`MathVista -> AI2D -> ChartQA -> MMMU -> TextVQA`

Outputs (one CSV per dataset, same schema as the earlier `scores_*_hybrid.csv` files
so they merge directly with existing results):
- `scores_ai2d_hybrid.csv`
- `scores_chartqa_hybrid.csv`
- `scores_textvqa_hybrid.csv`

In [ ]:
!pip install -q transformers accelerate sentence-transformers datasets pillow qwen-vl-utils

In [ ]:
import re, csv, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

K = 6
MAX_NEW_TOKENS = 20
TEMPERATURE = 1.1
TOP_P = 0.95
SHORT_ANSWER_TOKEN_THRESHOLD = 3
ENTROPY_RESCALE = 0.05

print("Loading Qwen2-VL-2B-Instruct (same model as the MathVista/MMMU pilot, for a fair comparison) ...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=DEVICE,
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
print("Models loaded.")

In [ ]:
# ------------------------------------------------------------------
# Dataset loaders -> unified item dicts:
#   {id, image (PIL), question, choices (list[str] or None), answers (list[str])}
# `answers` is always a LIST of acceptable normalized answers (TextVQA has
# multiple annotators; the others just have one, wrapped in a list for
# uniform handling).
# ------------------------------------------------------------------

def load_ai2d(max_examples):
    ds = load_dataset("lmms-lab/ai2d", split="test")
    items = []
    for i, ex in enumerate(ds):
        if i >= max_examples:
            break
        choices = ex["options"]
        letters = "ABCDEFGH"[:len(choices)]
        try:
            idx = int(ex["answer"])
            gt = letters[idx]   # FIXED: store the LETTER (matches how the
                                 # prompt asks the model to answer), not the
                                 # choice text -- storing text here caused a
                                 # near-total mismatch against the model's
                                 # lettered answers in the first run.
        except Exception:
            gt = str(ex["answer"])
        items.append({
            "id": f"ai2d_{i}",
            "image": ex["image"],
            "question": ex["question"],
            "choices": choices,
            "answers": [gt],
        })
    return items


def load_chartqa(max_examples):
    ds = load_dataset("HuggingFaceM4/ChartQA", split="test")
    items = []
    for i, ex in enumerate(ds):
        if i >= max_examples:
            break
        question = ex.get("query") or ex.get("question")
        label = ex.get("label") or ex.get("answer")
        if isinstance(label, list):
            answers = [str(a) for a in label]
        else:
            answers = [str(label)]
        items.append({
            "id": f"chartqa_{i}",
            "image": ex["image"],
            "question": question,
            "choices": None,   # open-ended
            "answers": answers,
        })
    return items


def load_textvqa(max_examples):
    ds = load_dataset("lmms-lab-encoder/textvqa", split="validation")
    items = []
    for i, ex in enumerate(ds):
        if i >= max_examples:
            break
        answers = ex.get("answers") or []
        if isinstance(answers, str):
            answers = [answers]
        answers = [str(a) for a in answers] if answers else ["unknown"]
        items.append({
            "id": f"textvqa_{i}",
            "image": ex["image"],
            "question": ex["question"],
            "choices": None,  # open-ended
            "answers": answers,
        })
    return items


LOADERS = {
    "ai2d": load_ai2d,
    "chartqa": load_chartqa,
    "textvqa": load_textvqa,
}
print("Loaders ready:", list(LOADERS.keys()))

In [ ]:
# ------------------------------------------------------------------
# Prompting, scoring (hybrid: semantic volume + first-token entropy),
# and answer matching -- same design as the MathVista/MMMU pipeline
# (Sec. 3.2 of the paper), generalized to a LIST of acceptable answers.
# ------------------------------------------------------------------

def build_prompt(ex):
    q = ex["question"]
    if ex["choices"]:
        letters = "ABCDEFGH"
        opts = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(ex["choices"]))
        return f"{q}\n{opts}\nAnswer with only the letter of the correct choice."
    return f"{q}\nAnswer as briefly as possible (a few words at most)."


def resize_image(img, max_side=384):
    w, h = img.size
    scale = max_side / max(w, h)
    if scale < 1.0:
        img = img.resize((max(1, int(w*scale)), max(1, int(h*scale))))
    return img.convert("RGB")


def prepare_inputs(image, prompt):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[resize_image(image)], return_tensors="pt").to(DEVICE)


def generate_k_samples(inputs, k):
    input_len = inputs["input_ids"].shape[1]
    samples = []
    for _ in range(k):
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=1.3,
            )
        out = processor.batch_decode(gen_ids[:, input_len:], skip_special_tokens=True)[0]
        samples.append(out.strip())
    return samples


def first_token_entropy(inputs):
    with torch.no_grad():
        out = model.generate(**inputs, do_sample=False, max_new_tokens=1,
                              output_scores=True, return_dict_in_generate=True)
    logits = out.scores[0][0]
    probs = torch.softmax(logits.float(), dim=-1)
    entropy = -(probs * torch.log(probs + 1e-12)).sum().item()
    return entropy / np.log(logits.shape[-1])


def semantic_volume_and_confidence(embeddings):
    k = embeddings.shape[0]
    if k < 2:
        return 0.0, 1.0
    sims = embeddings @ embeddings.T
    iu = np.triu_indices(k, k=1)
    V = float(np.mean(1.0 - sims[iu]))
    centroid = embeddings.mean(axis=0)
    centroid /= (np.linalg.norm(centroid) + 1e-8)
    c = float(np.clip((np.mean(embeddings @ centroid) + 1) / 2, 0, 1))
    return V, c


def normalize_answer(a):
    a = str(a).strip().lower()
    return re.sub(r"[^a-z0-9.\-]", "", a)


def majority_vote(answers):
    return Counter(normalize_answer(a) for a in answers).most_common(1)[0][0]


def is_correct(pred, gt_list):
    # TextVQA-style soft matching: correct if it matches ANY acceptable answer.
    pred_n = normalize_answer(pred)
    return int(any(pred_n == normalize_answer(g) for g in gt_list))

In [ ]:
# ------------------------------------------------------------------
# Main per-dataset loop (checkpointed: safe to interrupt and resume)
# ------------------------------------------------------------------

def run_dataset(dataset_key, max_examples=400):
    out_path = f"scores_{dataset_key}_hybrid.csv"
    header = ["id", "order", "nonconformity_score", "score_type",
              "correct", "majority_answer", "gt_answer", "samples"]

    done_ids = set()
    if Path(out_path).exists():
        with open(out_path) as f:
            r = csv.reader(f); next(r, None)
            done_ids = {row[0] for row in r if row}
        print(f"Resuming {dataset_key}: {len(done_ids)} already done.")
    else:
        with open(out_path, "w", newline="") as f:
            csv.writer(f).writerow(header)

    print(f"Loading dataset: {dataset_key} (max {max_examples}) ...")
    examples = LOADERS[dataset_key](max_examples)
    examples = [e for e in examples if e["id"] not in done_ids]
    print(f"{len(examples)} examples remaining.")

    f = open(out_path, "a", newline="")
    writer = csv.writer(f)
    t0 = time.time()

    for idx, ex in enumerate(examples):
        try:
            prompt = build_prompt(ex)
            inputs = prepare_inputs(ex["image"], prompt)
            samples = generate_k_samples(inputs, K)
            avg_len = np.mean([len(processor.tokenizer(s).input_ids) for s in samples])

            emb = embedder.encode(samples, normalize_embeddings=True)
            V, c = semantic_volume_and_confidence(np.array(emb))
            vol_score = V * (1 - c)

            H1 = first_token_entropy(inputs)
            entropy_score = H1 * ENTROPY_RESCALE

            if avg_len <= SHORT_ANSWER_TOKEN_THRESHOLD:
                score, score_type = entropy_score, "entropy"
            else:
                score, score_type = vol_score, "volume"

            maj = majority_vote(samples)
            correct = is_correct(maj, ex["answers"])
            gt_display = ex["answers"][0]

            writer.writerow([ex["id"], idx, score, score_type, correct, maj, gt_display, "|".join(samples)])
            f.flush()

            if idx < 3:
                print(f"  [{ex['id']}] avg_len={avg_len:.1f} -> {score_type} score={score:.4f} "
                      f"correct={correct} samples={samples}")

        except Exception as e:
            print(f"error on {ex['id']}: {e}")
            continue

        if idx % 20 == 0:
            elapsed = time.time() - t0
            rate = elapsed / (idx + 1)
            eta_min = rate * (len(examples) - idx) / 60
            print(f"[{dataset_key}] {idx+1}/{len(examples)} | {rate:.1f}s/ex | ETA {eta_min:.1f} min", flush=True)

    f.close()
    print(f"Done: {out_path}")
    return out_path

## Run each dataset

Start with a small `max_examples` (e.g. 10) to sanity-check timing, then increase.
Each call is checkpointed -- interrupt anytime and rerun the same cell to resume.

In [ ]:
run_dataset("ai2d", max_examples=400)

In [ ]:
run_dataset("chartqa", max_examples=400)

In [ ]:
run_dataset("textvqa", max_examples=400)

## Download the results

**Kaggle:** the three CSV files appear in the "Output" panel on the right side of the
notebook editor (or under `/kaggle/working/`) once the run finishes -- download them
from there.

**Colab:** run the cell below.

In [ ]:
# Colab only -- uncomment to use:
# from google.colab import files
# for name in ["scores_ai2d_hybrid.csv", "scores_chartqa_hybrid.csv", "scores_textvqa_hybrid.csv"]:
#     files.download(name)